In [ ]:
# Install LeRobot with SmolVLA dependencies
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"
!pip install -q num2words

In [ ]:
import torch
import numpy as np
import inspect
import textwrap
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
policy = policy.to(device).eval()
print("Model loaded.")

In [ ]:
import torch
import numpy as np

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.utils import prepare_observation_for_inference

MODEL_PATH = "lerobot/smolvla_base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy = SmolVLAPolicy.from_pretrained(MODEL_PATH)
policy = policy.to(DEVICE).eval()
policy.requires_grad_(False)

preprocessor, _ = make_pre_post_processors(
    policy.config,
    pretrained_path=MODEL_PATH,
    preprocessor_overrides={"device_processor": {"device": str(DEVICE)}},
    postprocessor_overrides={"device_processor": {"device": str(DEVICE)}},
)

print("Model loaded.")

In [ ]:
dummy_obs = prepare_observation_for_inference(
    {
        "observation.state": np.zeros((8,), dtype=np.float32),
        "observation.images.camera1": np.zeros((512, 512, 3), dtype=np.uint8),
        "observation.images.camera2": np.zeros((512, 512, 3), dtype=np.uint8),
    },
    DEVICE,
    task="pick up the red block",
)
dummy_obs = preprocessor(dummy_obs)

print("Keys after preprocessing:")
for k, v in dummy_obs.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: shape={list(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
model = policy.model  # VLAFlowMatching

with torch.inference_mode():
    # Use the model's own predict_action_chunk path but intercept.
    # Easiest: call sample_actions internals directly.
    images, img_masks = policy.prepare_images(dummy_obs)
    state = policy.prepare_state(dummy_obs)

    # Find the right key names from Cell 2 output
    lang_key = [k for k in dummy_obs if "language_token" in k.lower() or "lang" in k.lower()]
    mask_key = [k for k in dummy_obs if "language" in k.lower() and "mask" in k.lower() or "attention_mask" in k.lower()]
    print(f"Language keys found: {lang_key}")
    print(f"Mask keys found: {mask_key}")

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks

# Monkey-patch to capture past_key_values
_captured_cache = [None]
_orig_denoise_step = model.denoise_step

def _patched_denoise_step(prefix_pad_masks, past_key_values, x_t, timestep):
    if _captured_cache[0] is None:
        _captured_cache[0] = past_key_values
        print("Cache captured on first denoise step!")
    return _orig_denoise_step(prefix_pad_masks, past_key_values, x_t, timestep)

model.denoise_step = _patched_denoise_step

with torch.inference_mode():
    # Run full forward — this calls sample_actions internally
    _ = policy.predict_action_chunk(dummy_obs)

model.denoise_step = _orig_denoise_step  # restore

past_key_values = _captured_cache[0]
print(f"\nCache type: {type(past_key_values).__name__}")

In [ ]:
pkv = past_key_values

if hasattr(pkv, 'key_cache'):
    print(f"DynamicCache with {len(pkv.key_cache)} layers\n")
    total_bytes = 0
    for i, (k, v) in enumerate(zip(pkv.key_cache, pkv.value_cache)):
        kb = k.nelement() * k.element_size()
        vb = v.nelement() * v.element_size()
        total_bytes += kb + vb
        print(f"Layer {i:2d}: K={list(k.shape)} {k.dtype}  "
              f"V={list(v.shape)} {v.dtype}  "
              f"({(kb + vb) / 1024:.1f} KB)")
    print(f"\nTotal: {total_bytes / 1024:.1f} KB = {total_bytes / (1024**2):.2f} MB")

elif isinstance(pkv, (tuple, list)):
    print(f"Tuple cache with {len(pkv)} entries\n")
    for i, entry in enumerate(pkv):
        if isinstance(entry, (tuple, list)):
            for j, t in enumerate(entry):
                print(f"Layer {i:2d}[{j}]: shape={list(t.shape)}, dtype={t.dtype}")
        elif isinstance(entry, torch.Tensor):
            print(f"Layer {i:2d}: shape={list(entry.shape)}, dtype={entry.dtype}")
        else:
            print(f"Layer {i:2d}: {type(entry)}")
else:
    print(f"Unknown: {type(pkv)}")
    print(dir(pkv))

In [ ]:
pkv = past_key_values

print(f"Type: {type(pkv)}")
print(f"Keys: {list(pkv.keys())}")
print(f"Num entries: {len(pkv)}")

for key in list(pkv.keys())[:3]:
    val = pkv[key]
    print(f"\n  pkv[{key!r}]:")
    print(f"    type: {type(val).__name__}")
    if isinstance(val, torch.Tensor):
        print(f"    shape: {list(val.shape)}, dtype: {val.dtype}")
    elif isinstance(val, (tuple, list)):
        print(f"    len: {len(val)}")
        for j, t in enumerate(val):
            if isinstance(t, torch.Tensor):
                print(f"    [{j}]: shape={list(t.shape)}, dtype={t.dtype}")
            else:
                print(f"    [{j}]: {type(t).__name__}")
    elif isinstance(val, dict):
        print(f"    keys: {list(val.keys())[:5]}")
    elif hasattr(val, 'key_cache'):
        print(f"    DynamicCache! len(key_cache)={len(val.key_cache)}")
        for j, (k, v) in enumerate(zip(val.key_cache, val.value_cache)):
            print(f"    layer {j}: K={list(k.shape)} {k.dtype}, V={list(v.shape)} {v.dtype}")
    else:
        print(f"    repr: {repr(val)[:200]}")

In [ ]:
pkv = past_key_values

total_bytes = 0
print(f"{'Layer':>5} {'K shape':>25} {'K dtype':>10} {'V shape':>25} {'V dtype':>10} {'KB':>8}")
for i in range(16):
    k = pkv[i]['key_states']
    v = pkv[i]['value_states']
    kb = k.nelement() * k.element_size()
    vb = v.nelement() * v.element_size()
    total_bytes += kb + vb
    print(f"{i:5d} {str(list(k.shape)):>25} {str(k.dtype):>10} "
          f"{str(list(v.shape)):>25} {str(v.dtype):>10} {(kb+vb)/1024:8.1f}")

print(f"\nTotal: {total_bytes/1024:.1f} KB = {total_bytes/(1024**2):.2f} MB")

In [ ]:
print(f"{'Layer':>5} {'K_min':>8} {'K_max':>8} {'K_std':>8} {'V_min':>8} {'V_max':>8} {'V_std':>8}")
for i in range(16):
    k = pkv[i]['key_states'].float()
    v = pkv[i]['value_states'].float()
    if k.numel() == 0:
        print(f"{i:5d}  (empty)")
        continue
    print(f"{i:5d} {k.min():8.4f} {k.max():8.4f} {k.std():8.4f} "
          f"{v.min():8.4f} {v.max():8.4f} {v.std():8.4f}")

# KIVI analysis on first layer
k0 = pkv[0]['key_states'].float()
v0 = pkv[0]['value_states'].float()
print(f"\nKIVI ANALYSIS (layer 0, K shape {list(k0.shape)}):")
print(f"  K per-channel std (across tokens, dim=-2): {k0.std(dim=-2).mean():.4f}")
print(f"  K per-token std (across head_dim, dim=-1):  {k0.std(dim=-1).mean():.4f}")
print(f"  V per-channel std (across tokens, dim=-2): {v0.std(dim=-2).mean():.4f}")
print(f"  V per-token std (across head_dim, dim=-1):  {v0.std(dim=-1).mean():.4f}")

In [ ]:
# Count how many times each cache layer is accessed during denoise_step
_access_counts = {i: 0 for i in range(16)}
_orig_getitem = past_key_values.__class__.__getitem__

class AccessTracker(dict):
    def __getitem__(self, key):
        if isinstance(key, int):
            _access_counts[key] += 1
        return super().__getitem__(key)

# Wrap the cache
tracked_cache = AccessTracker(past_key_values)

# Run ONE denoise step manually
with torch.inference_mode():
    noise = model.sample_noise((1, model.config.chunk_size, model.config.max_action_dim), DEVICE)
    timestep = torch.tensor(1.0, device=DEVICE).expand(1)

    images, img_masks = policy.prepare_images(dummy_obs)
    state = policy.prepare_state(dummy_obs)
    lang_tokens = dummy_obs["observation.language.tokens"]
    lang_masks = dummy_obs["observation.language.attention_mask"]

    prefix_embs, prefix_pad_masks, prefix_att_masks = model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state
    )

    _ = model.denoise_step(
        prefix_pad_masks=prefix_pad_masks,
        past_key_values=tracked_cache,
        x_t=noise,
        timestep=timestep,
    )

print("Cache layer access counts (1 denoise step):")
for i in range(16):
    marker = " <-- cross-attn" if _access_counts[i] > 0 else ""
    print(f"  Layer {i:2d}: {_access_counts[i]} accesses{marker}")

total_accessed = sum(1 for v in _access_counts.values() if v > 0)
print(f"\n{total_accessed} of 16 layers accessed")

In [ ]:
def quantize_per_token_int4(tensor):
    """Quantize BF16 tensor to INT4 with per-token scale/zero-point.

    Input:  [B, S, H, D] in BF16
    Output: (quantized_int8, scales, zeros) where:
      - quantized_int8: [B, S, H, D] in int8 (values 0-15)
      - scales: [B, S, H, 1] in float32
      - zeros:  [B, S, H, 1] in float32

    We store as int8 (not packed) for simplicity. A production
    implementation would bit-pack two INT4 values per byte.
    """
    x = tensor.float()  # [B, S, H, D]

    # Per-token min/max along head_dim (last axis)
    x_min = x.amin(dim=-1, keepdim=True)  # [B, S, H, 1]
    x_max = x.amax(dim=-1, keepdim=True)  # [B, S, H, 1]

    # INT4 range: 0 to 15
    qmin, qmax = 0, 15

    # Scale and zero-point (asymmetric quantization)
    scale = (x_max - x_min) / (qmax - qmin)
    scale = scale.clamp(min=1e-8)  # avoid division by zero
    zero = x_min - scale * qmin

    # Quantize
    x_q = ((x - zero) / scale).round().clamp(qmin, qmax).to(torch.int8)

    return x_q, scale, zero


def dequantize_per_token_int4(x_q, scale, zero):
    """Dequantize INT4 back to BF16.

    Returns tensor in the original shape and BF16 dtype.
    """
    return (x_q.float() * scale + zero).to(torch.bfloat16)


# === Test on the actual cache ===
total_mse_k = 0.0
total_mse_v = 0.0

for i in range(16):
    k_orig = pkv[i]['key_states']
    v_orig = pkv[i]['value_states']

    # Quantize + dequantize
    kq, ks, kz = quantize_per_token_int4(k_orig)
    vq, vs, vz = quantize_per_token_int4(v_orig)
    k_recon = dequantize_per_token_int4(kq, ks, kz)
    v_recon = dequantize_per_token_int4(vq, vs, vz)

    k_mse = (k_orig.float() - k_recon.float()).pow(2).mean().item()
    v_mse = (v_orig.float() - v_recon.float()).pow(2).mean().item()
    total_mse_k += k_mse
    total_mse_v += v_mse

    if i < 3 or i == 15:
        print(f"Layer {i:2d}: K_MSE={k_mse:.6f}  V_MSE={v_mse:.6f}")

print(f"\nAvg K reconstruction MSE: {total_mse_k/16:.6f}")
print(f"Avg V reconstruction MSE: {total_mse_v/16:.6f}")

# Memory comparison
orig_bytes = sum(pkv[i]['key_states'].nelement() * 2 + pkv[i]['value_states'].nelement() * 2 for i in range(16))
# Quantized: int8 (1 byte) + scale/zero (4 bytes each per token-head)
q_data_bytes = sum(pkv[i]['key_states'].nelement() * 1 + pkv[i]['value_states'].nelement() * 1 for i in range(16))
q_meta_bytes = sum(pkv[i]['key_states'][..., :1].nelement() * 8 + pkv[i]['value_states'][..., :1].nelement() * 8 for i in range(16))
print(f"\nOriginal:   {orig_bytes/1024:.1f} KB (BF16)")
print(f"Quantized:  {(q_data_bytes + q_meta_bytes)/1024:.1f} KB (INT4 unpacked + scales)")
print(f"Compression: {orig_bytes / (q_data_bytes + q_meta_bytes):.2f}x")

In [ ]:
def quantize_cache(pkv, key_bits=4, value_bits=4):
    """Quantize all 16 layers of the KV cache dict.
    Returns a new dict with quantized+dequantized values (simulated).
    """
    q_pkv = {}
    for i in range(16):
        k_orig = pkv[i]['key_states']
        v_orig = pkv[i]['value_states']

        if key_bits == 4:
            kq, ks, kz = quantize_per_token_int4(k_orig)
            k_recon = dequantize_per_token_int4(kq, ks, kz)
        else:
            # INT8: 256 levels
            k_recon = _quantize_dequantize(k_orig, bits=key_bits)

        if value_bits == 4:
            vq, vs, vz = quantize_per_token_int4(v_orig)
            v_recon = dequantize_per_token_int4(vq, vs, vz)
        else:
            v_recon = _quantize_dequantize(v_orig, bits=value_bits)

        q_pkv[i] = {'key_states': k_recon, 'value_states': v_recon}

    return q_pkv


def _quantize_dequantize(tensor, bits=8):
    """Generic per-token symmetric quantize+dequantize."""
    x = tensor.float()
    qmax = (1 << (bits - 1)) - 1  # 127 for INT8
    qmin = -qmax

    x_absmax = x.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8)
    scale = x_absmax / qmax

    x_q = (x / scale).round().clamp(qmin, qmax)
    return (x_q * scale).to(torch.bfloat16)


# === Run full sample_actions with original vs quantized cache ===

# We need to intercept the cache between VLM forward and ODE loop.
# Monkey-patch sample_actions to insert quantization.

import types
from lerobot.policies.smolvla.modeling_smolvla import make_att_2d_masks

def sample_actions_with_quant(self, images, img_masks, lang_tokens, lang_masks,
                              state, noise=None, key_bits=4, value_bits=4, **kwargs):
    """Modified sample_actions that quantizes KV cache before ODE loop."""
    bsize = state.shape[0]
    device = state.device

    if noise is None:
        actions_shape = (bsize, self.config.chunk_size, self.config.max_action_dim)
        noise = self.sample_noise(actions_shape, device)

    prefix_embs, prefix_pad_masks, prefix_att_masks = self.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state
    )
    prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1

    _, past_key_values = self.vlm_with_expert.forward(
        attention_mask=prefix_att_2d_masks,
        position_ids=prefix_position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=self.config.use_cache,
        fill_kv_cache=True,
    )

    # === QUANTIZE THE CACHE HERE ===
    if key_bits < 16 or value_bits < 16:
        past_key_values = quantize_cache(past_key_values, key_bits, value_bits)

    num_steps = self.config.num_steps
    dt = -1.0 / num_steps
    x_t = noise

    for step in range(num_steps):
        time = 1.0 + step * dt
        time_tensor = torch.tensor(time, dtype=torch.float32, device=device).expand(bsize)
        v_t = self.denoise_step(
            x_t=x_t,
            prefix_pad_masks=prefix_pad_masks,
            past_key_values=past_key_values,
            timestep=time_tensor,
        )
        x_t = x_t + dt * v_t

    return x_t


# === Compare: original vs INT4/INT4 vs INT8-K/INT4-V ===
with torch.inference_mode():
    images, img_masks = policy.prepare_images(dummy_obs)
    state = policy.prepare_state(dummy_obs)
    lang_tokens = dummy_obs["observation.language.tokens"]
    lang_masks = dummy_obs["observation.language.attention_mask"]

    # Use the SAME noise for fair comparison
    noise = model.sample_noise(
        (1, model.config.chunk_size, model.config.max_action_dim), DEVICE
    )

    # Original (no quantization)
    actions_orig = sample_actions_with_quant(
        model, images, img_masks, lang_tokens, lang_masks, state,
        noise=noise.clone(), key_bits=16, value_bits=16
    )

    # INT4 keys / INT4 values (aggressive)
    actions_int4_int4 = sample_actions_with_quant(
        model, images, img_masks, lang_tokens, lang_masks, state,
        noise=noise.clone(), key_bits=4, value_bits=4
    )

    # INT8 keys / INT4 values (KIVI-style)
    actions_int8k_int4v = sample_actions_with_quant(
        model, images, img_masks, lang_tokens, lang_masks, state,
        noise=noise.clone(), key_bits=8, value_bits=4
    )

    # INT4 keys / INT8 values (inverted — for comparison)
    actions_int4k_int8v = sample_actions_with_quant(
        model, images, img_masks, lang_tokens, lang_masks, state,
        noise=noise.clone(), key_bits=4, value_bits=8
    )

# Compute action-level MSE (what the accuracy gate cares about)
orig_f = actions_orig.float()
mse_44 = (orig_f - actions_int4_int4.float()).pow(2).mean().item()
mse_84 = (orig_f - actions_int8k_int4v.float()).pow(2).mean().item()
mse_48 = (orig_f - actions_int4k_int8v.float()).pow(2).mean().item()

print("Action-level MSE (vs unquantized, same noise):")
print(f"  INT4-K / INT4-V:  {mse_44:.6f}")
print(f"  INT8-K / INT4-V:  {mse_84:.6f}  (KIVI-style)")
print(f"  INT4-K / INT8-V:  {mse_48:.6f}  (inverted)")
print()
print(f"Baseline action MSE (vs ground truth): 0.954")
print(f"Accuracy gate:  MSE <= 1.431")
print(f"Budget for quant error: {1.431 - 0.954:.3f}")
print()
print("Verdict:")
for name, mse in [("INT4/INT4", mse_44), ("INT8-K/INT4-V", mse_84), ("INT4-K/INT8-V", mse_48)]:
    pct = mse / 0.954 * 100
    print(f"  {name}: adds {mse:.4f} to MSE ({pct:.2f}% of baseline) "
          f"{'SAFE' if mse < 0.2 else 'RISKY' if mse < 0.477 else 'DANGEROUS'}")

In [ ]:
import time

NUM_RUNS = 20  # enough to get stable timing

def run_sample_actions(model, obs, noise, key_bits=16, value_bits=16):
    """Run one full forward pass, optionally with KV cache quantization."""
    images, img_masks = policy.prepare_images(obs)
    state = policy.prepare_state(obs)
    lang_tokens = obs["observation.language.tokens"]
    lang_masks = obs["observation.language.attention_mask"]

    return sample_actions_with_quant(
        model, images, img_masks, lang_tokens, lang_masks, state,
        noise=noise.clone(), key_bits=key_bits, value_bits=value_bits
    )


def benchmark(label, key_bits, value_bits, num_runs=NUM_RUNS):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    # Fresh noise each run but same across configs
    base_noise = model.sample_noise(
        (1, model.config.chunk_size, model.config.max_action_dim), DEVICE
    )

    # Warmup
    with torch.inference_mode():
        for _ in range(3):
            _ = run_sample_actions(model, dummy_obs, base_noise, key_bits, value_bits)
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    latencies = []
    with torch.inference_mode():
        for _ in range(num_runs):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = run_sample_actions(model, dummy_obs, base_noise, key_bits, value_bits)
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)

    peak_mem = torch.cuda.max_memory_allocated() / (1024**3)
    mean_lat = sum(latencies) / len(latencies)
    p95_lat = sorted(latencies)[int(0.95 * len(latencies))]

    print(f"{label:>20s}: mean={mean_lat:.1f}ms  p95={p95_lat:.1f}ms  "
          f"peak_mem={peak_mem:.3f}GB  throughput={50/mean_lat*1000:.0f} act/s")
    return mean_lat, peak_mem


print("=== Standalone benchmark (no torch.compile) ===\n")
lat_orig, mem_orig = benchmark("BF16 (no quant)", 16, 16)
lat_44, mem_44 = benchmark("INT4-K / INT4-V", 4, 4)
lat_84, mem_84 = benchmark("INT8-K / INT4-V", 8, 4)

print(f"\n=== Delta vs unquantized ===")
print(f"  INT4/INT4: latency {(lat_44/lat_orig - 1)*100:+.1f}%  memory {(mem_44/mem_orig - 1)*100:+.1f}%")
print(f"  INT8/INT4: latency {(lat_84/lat_orig - 1)*100:+.1f}%  memory {(mem_84/mem_orig - 1)*100:+.1f}%")